# EuroSAT Land-Use Classification with a CNN (PyTorch)

**Objective:** train a convolutional neural network to classify 64×64 RGB satellite images into 10 land-cover classes.

Dataset: [`tanganke/eurosat`](https://huggingface.co/datasets/tanganke/eurosat)

This notebook creates a stratified validation split from the supplied training set, trains a CNN, evaluates once on the supplied test set, and saves the model and evaluation artifacts.



## 1. Install dependencies
Run this cell in a fresh Colab or Jupyter environment.

In [ ]:
%pip install -q torch torchvision datasets huggingface_hub scikit-learn matplotlib seaborn pandas tqdm

## 2. Imports and reproducibility

In [ ]:
import os, random, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 3. Load and inspect the dataset

In [ ]:
DATASET_ID = "tanganke/eurosat"
raw = load_dataset(DATASET_ID)

print(raw)
print("Train examples:", len(raw["train"]))
print("Test examples:", len(raw["test"]))
print("Features:", raw["train"].features)

class_names = raw["train"].features["label"].names
num_classes = len(class_names)
print("\nClass names:")
for i, name in enumerate(class_names):
    print(i, name)

# Create a reproducible, class-stratified validation split.
split = raw["train"].train_test_split(
    test_size=0.20,
    seed=SEED,
    stratify_by_column="label"
)
train_hf = split["train"]
val_hf = split["test"]
test_hf = raw["test"]

print(f"\nTrain: {len(train_hf):,} | Validation: {len(val_hf):,} | Test: {len(test_hf):,}")

## 4. Explore class distribution and sample images

In [ ]:
counts = pd.Series(train_hf["label"]).value_counts().sort_index()
distribution = pd.DataFrame({
    "class_id": range(num_classes),
    "class_name": class_names,
    "training_images": counts.values
})
display(distribution)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for class_id, ax in enumerate(axes.flat):
    idx = next(i for i, y in enumerate(train_hf["label"]) if y == class_id)
    ax.imshow(train_hf[idx]["image"].convert("RGB"))
    ax.set_title(class_names[class_id])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Preprocessing and data augmentation

In [ ]:
IMAGE_SIZE = 64
BATCH_SIZE = 64
NUM_WORKERS = 2

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class EuroSATDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[index]
        image = row["image"].convert("RGB")
        image = self.transform(image)
        label = int(row["label"])
        return image, label

train_ds = EuroSATDataset(train_hf, train_transform)
val_ds = EuroSATDataset(val_hf, eval_transform)
test_ds = EuroSATDataset(test_hf, eval_transform)

pin_memory = DEVICE.type == "cuda"
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin_memory
)

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)  # [batch, 3, 64, 64]
print("Batch label shape:", labels.shape)

## 6. Define the CNN

In [ ]:
class EuroSAT_CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 3 x 64 x 64 -> 32 x 32 x 32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2: 32 x 32 x 32 -> 64 x 16 x 16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3: 64 x 16 x 16 -> 128 x 8 x 8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = EuroSAT_CNN(num_classes=num_classes).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. Configure training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

EPOCHS = 25
EARLY_STOPPING_PATIENCE = 6

## 8. Training and validation loop
The best checkpoint is selected using validation accuracy. Early stopping prevents unnecessary epochs when validation accuracy stops improving.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(x)
            loss = criterion(logits, y)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


history = {
    "train_loss": [], "train_accuracy": [],
    "val_loss": [], "val_accuracy": []
}
best_val_accuracy = 0.0
best_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)

    scheduler.step(val_acc)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f}, acc {val_acc:.4f}"
    )

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        torch.save(best_state, "eurosat_cnn_best.pth")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("Early stopping.")
        break

model.load_state_dict(best_state)
print(f"Training time: {(time.time() - start_time) / 60:.1f} minutes")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")

## 9. Visualize training history

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(epochs_ran, history["train_loss"], label="Train")
axes[0].plot(epochs_ran, history["val_loss"], label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].legend()

axes[1].plot(epochs_ran, history["train_accuracy"], label="Train")
axes[1].plot(epochs_ran, history["val_accuracy"], label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 10. Final evaluation on the held-out test set

The test set is evaluated only after model selection. The report includes per-class precision, recall, F1-score, and support. Macro averages give each class equal weight; weighted averages account for class support.

In [ ]:
model.eval()
all_targets, all_predictions = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        predictions = logits.argmax(dim=1).cpu().numpy()
        all_predictions.extend(predictions)
        all_targets.extend(y.numpy())

test_accuracy = accuracy_score(all_targets, all_predictions)
print(f"Test accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print("\nClassification report:")
print(classification_report(
    all_targets, all_predictions,
    labels=list(range(num_classes)),
    target_names=class_names,
    digits=4,
    zero_division=0
))

## 11. Confusion matrix

In [ ]:
cm = confusion_matrix(all_targets, all_predictions, labels=list(range(num_classes)))
plt.figure(figsize=(11, 9))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("EuroSAT test-set confusion matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Optional robustness evaluation

In [ ]:
corruption_splits = [
    "contrast", "gaussian_noise", "impulse_noise",
    "jpeg_compression", "motion_blur", "pixelate", "spatter"
]

robustness_results = []
for split_name in corruption_splits:
    if split_name not in raw:
        print(f"Split {split_name!r} not available; skipping.")
        continue

    loader = DataLoader(
        EuroSATDataset(raw[split_name], eval_transform),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=pin_memory
    )
    targets, predictions = [], []
    model.eval()
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(DEVICE, non_blocking=True))
            predictions.extend(logits.argmax(1).cpu().numpy())
            targets.extend(y.numpy())

    score = accuracy_score(targets, predictions)
    robustness_results.append({"split": split_name, "accuracy": score})
    print(f"{split_name:20s}: {score:.4f}")

robustness_df = pd.DataFrame(robustness_results)
display(robustness_df)

## 13. Save model and experiment outputs

In [ ]:
checkpoint = {
    "model_state_dict": model.state_dict(),
    "class_names": class_names,
    "image_size": IMAGE_SIZE,
    "normalization_mean": mean,
    "normalization_std": std,
    "test_accuracy": float(test_accuracy),
    "best_validation_accuracy": float(best_val_accuracy),
}
torch.save(checkpoint, "eurosat_cnn_checkpoint.pth")

pd.DataFrame(history).to_csv("training_history.csv", index=False)
pd.DataFrame(
    classification_report(
        all_targets, all_predictions,
        labels=list(range(num_classes)),
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )
).transpose().to_csv("classification_report.csv")

pd.DataFrame(cm, index=class_names, columns=class_names).to_csv("confusion_matrix.csv")

if len(robustness_results):
    robustness_df.to_csv("robustness_results.csv", index=False)

print("Saved: eurosat_cnn_best.pth")
print("Saved: eurosat_cnn_checkpoint.pth")
print("Saved: training_history.csv, classification_report.csv, confusion_matrix.csv")